# QueueWise — Dua Kasir & Estimasi Waktu sampai Selesai
**Mata kuliah:** Computer Vision · **Tugas:** Week 8 — Progress Submission & Week 9 — Final Demo  
**Kelompok:** _isi_ · **Anggota:** _Nama – NIM_

Notebook revisi: **Kasir Tengah + Kasir Kanan**, masing-masing **satu garis masuk dan satu garis keluar**. Posisi masuk tetap; **garis keluar diperlebar** setelah review cuplikan video output `download (1).mp4`.

**Perbaikan exit terbaru:** garis keluar tiap kasir **dipindah mendekati posisi akhir kaki pelanggan** (Tengah y≈298–310 → 240–252; Kanan y≈260–285 → 206–222), **dilebarkan** (Tengah x=330–470, Kanan x=520–639) dan **dimiringkan mengikuti lantai**. Vektor `after` berada tepat di sisi luar (bawah) garis. `queue_roi` diperluas sampai kedua ujung garis exit berada di dalam zona pelanggan. `SEGMENT_MARGIN_PX` dinaikkan 15 → 30; `EXIT_DEADBAND_PX` tetap 2 px karena menaikkannya justru menurunkan jumlah exit yang terdeteksi.

**Cara menjalankan di Google Colab**
1. Buka notebook ini, pilih runtime GPU jika tersedia.
2. Pilih **Runtime → Restart session and run all**. Saat diminta, unggah **video asli** kasir.
3. Periksa preview: titik merah adalah **titik kaki**; pelanggan berada di zona cyan, petugas di luar zona. Zona ungu menandai area layanan.
4. Tunggu pipeline selesai. Bagian terakhir memutar video dashboard dan menyediakan ZIP hasil.

**Yang tampil di dashboard:** jumlah menunggu, sedang dilayani, masuk/keluar, estimasi mulai dilayani, dan **estimasi waktu sampai selesai jika bergabung sekarang**. Angka awal diberi label **asumsi awal/manual**; setelah sampel cukup, sumber berubah menjadi **data kasir**.

**Status verifikasi revisi:** 60 tes logika/geometri lulus, termasuk lintasan samping yang lolos garis lama, jitter exit, ID baru dekat exit, dan ID di area petugas. Ekspor CSV, grafik, dan evaluasi diuji dengan simulasi antrean serta kondisi tanpa event. Cuplikan video output 1760×720 ditinjau; inferensi YOLO ulang pada video asli belum dijalankan. Jalankan ulang menggunakan rekaman asli, bukan video output berpanel.


## 1. Instalasi dan import
Cell instalasi memeriksa dependensi melalui pip. Jika GPU tidak tersedia, inferensi tetap dapat berjalan di CPU, tetapi lebih lambat. Internet diperlukan untuk instalasi awal dan pengunduhan bobot YOLO.


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics>=8.3.161,<9", "lap>=0.5", "opencv-python>=4.8",
    "numpy>=1.24", "pandas>=2", "matplotlib>=3.7", "PyYAML>=6",
])


In [ ]:
import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO
from IPython.display import display

print("Python", sys.version.split()[0], "| OpenCV", cv2.__version__, "| Ultralytics", ultralytics.__version__)
print("GPU tersedia:", torch.cuda.is_available())


## 2. Konfigurasi video, dua lane, zona, dan waktu
Koordinat `LANES` memakai referensi **640×360**. Garis/zona otomatis diskalakan bila resolusi video berbeda; sudut kamera tetap harus sama. **TENGAH MASUK** kembali ke posisi awal; **KANAN MASUK** berada di posisi kanan atas hasil revisi.

- `queue_roi`: koridor pelanggan berdasarkan **titik kaki**, untuk memisahkan pelanggan dan petugas.
- `service_roi`: area tempat pelanggan paling depan dianggap mulai dilayani setelah terlihat stabil. Ini pendekatan visual; waktu sampai garis keluar juga mencakup gerakan meninggalkan kasir.
- `BOOTSTRAP_S`: orang yang sudah berada dalam zona saat video dimulai didaftarkan sebagai `INITIAL`, bukan crossing masuk palsu.
- `RECOVERY_ENABLED`: pelanggan yang baru terlihat setelah garis masuk dapat didaftarkan setelah stabil dalam zona; dicatat `ENTRY_RECOVERED` dan dipisahkan dari crossing garis.
- `RECOVERY_EXIT_GUARD_PX`: tidak membuat registrasi baru dekat pintu keluar; pelanggan yang sudah terdaftar tetap boleh keluar.
- `OUTSIDE_ROI_TIMEOUT_S`: ID yang terus terlihat di area petugas tidak menahan antrean selamanya. Area mendekati ruas keluar dikecualikan dari timeout ini.
- `SERVICE_PRIOR_RANGE_S = (25,45,90)` adalah **asumsi awal detik per orang**, bukan hasil pengukuran video. Ganti sesuai konteks, atau isi `None` untuk menunggu kalibrasi aktual.

Untuk uji cepat, isi `MAX_FRAMES = 300`; untuk rekaman lengkap, gunakan `None`.


In [ ]:
VIDEO_PATH = Path("/content/vidssave.com HD-SDI 1080P Hi Definition Megapixel Security Camera @ Super Market Checkout Sample #1 360P.mp4")
REFERENCE_SIZE = (640, 360)
MODEL_NAME = "yolo11s.pt"  # Lebih kuat untuk orang kecil/tertutup dibanding yolo11n.
IMGSZ = 960
CONF = 0.15
IOU = 0.50
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TRACKER_TYPE = "botsort"
USE_REID = True  # Otomatis mencoba ByteTrack jika konfigurasi ini tidak didukung.

# Dua kasir, tepat satu garis masuk per kasir. Posisi garis sesuai revisi terakhir.
LANES = {
    "Kasir Tengah": {
        "entry": [{"p1": (325, 25), "p2": (415, 25), "after": (380, 140)}],
        # EXIT direvisi: lebih pendek jaraknya dari tempat bayar (y 298-310 -> 240-252), lebih lebar (x 330-470),
        # miring searah lantai konter, dan 'after' tepat di sisi luar (bawah) garis, searah orang yang pergi.
        "exit": {"p1": (330, 252), "p2": (470, 240), "after": (400, 290)},
        # queue_roi diperluas ke bawah/samping sampai KEDUA ujung exit + margin berada di dalam zona pelanggan.
        "queue_roi": [(335, 15), (416, 15), (432, 145), (450, 205), (474, 232), (474, 285), (325, 285), (325, 240), (348, 188)],
        "service_roi": [(350, 105), (418, 100), (438, 225), (358, 232)],
    },
    "Kasir Kanan": {
        "entry": [{"p1": (485, 39), "p2": (530, 32), "after": (505, 90)}],
        "exit": {"p1": (520, 222), "p2": (639, 206), "after": (580, 250)},
        "queue_roi": [(480, 15), (540, 15), (600, 180), (639, 190), (639, 285), (575, 285), (512, 238), (528, 196), (510, 92)],
        "service_roi": [(532, 115), (595, 112), (620, 225), (550, 228)],
    },
}

DEADBAND_PX = 4.0
EXIT_DEADBAND_PX = 2.0  # Sengaja TIDAK dinaikkan: pita mati lebih lebar membuat crossing exit sulit terkonfirmasi (diuji: 2 px terbaik).
SEGMENT_MARGIN_PX = 30.0  # Toleransi ujung ruas (naik dari 15) agar orang yang lewat dari pinggir tetap terdeteksi; bukan perluasan ke seluruh gambar.
CROSSING_MAX_GAP_S = 6.0
LOST_TIMEOUT_S = 15.0
OUTSIDE_ROI_TIMEOUT_S = 3.0  # ID yang berpindah ke area petugas tidak menggantung dalam antrean.
MERGE_S = 3.0
MERGE_PX = 45.0
BOOTSTRAP_S = 2.0
RECOVERY_ENABLED = True
RECOVERY_STABLE_S = 0.8
RECOVERY_EXIT_GUARD_PX = 35.0  # Jangan daftarkan ID baru yang pertama muncul di dekat exit.
SERVICE_CONFIRM_S = 0.6

MIN_SAMPLES = 2
RECENT_N = 12
MIN_SERVICE_SAMPLE_S = 1.0
MAX_SERVICE_SAMPLE_S = 900.0
# Asumsi awal, BUKAN hasil pengukuran. Diganti data kasir setelah sampel cukup.
# Isi None jika ingin menampilkan "kalibrasi" sampai ada sampel aktual.
SERVICE_PRIOR_RANGE_S = (25.0, 45.0, 90.0)
USE_SHARED_SAMPLES = False  # Kecepatan dua kasir bisa berbeda.
WINDOW_S = 30

MAX_FRAMES = None  # Uji cepat: 300; demo lengkap: None.
OUT_SCALE = 2
PANEL_W = 480
SHOW_ROI = True
SHOW_TRAILS = True
SAVE_TRACK_LOG = True
OUT = Path("outputs_queuewise")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "screenshots").mkdir(exist_ok=True)


## 3. Logika crossing, pemulihan ID, dan waktu
**Mengapa versi lama bisa tetap nol:** orang dapat pertama kali terdeteksi ketika titik kakinya sudah di bawah garis masuk. Tanpa titik sebelumnya di sisi atas, tidak ada crossing yang bisa dikonfirmasi. ID yang terputus tepat di garis keluar juga dapat kehilangan riwayat sisi garis.

Revisi ini menyimpan titik stabil sebelum pita mati, memeriksa **titik potong lintasan** dengan ruas garis, mempertahankan riwayat pada ID yang digabung, serta menyediakan registrasi `INITIAL`/`ENTRY_RECOVERED` melalui zona pelanggan. Registrasi zona bukan bukti orang melewati garis; asalnya selalu dibedakan di log.

**Jumlah dalam sistem = masuk (garis + pemulihan) + awal − keluar − hilang.** Jumlah ini mencakup orang yang sedang dilayani. FIFO dipakai sebagai asumsi per kasir; satu orang dihitung masuk dan keluar paling banyak sekali.

**Estimasi untuk orang baru:**

`sampai mulai dilayani = sisa layanan orang di depan + jumlah menunggu × durasi layanan/orang`

`sampai selesai = sampai mulai dilayani + durasi layanannya sendiri`

Durasi layanan diperoleh dari pelanggan paling depan di zona layanan sampai garis keluar. Bila awal layanan terpotong, sampel penuh tidak dibuat; interval antar-keluar hanya dipakai ketika masih ada backlog. Durasi total antrean tidak dibagi panjang antrean untuk dianggap sebagai waktu layanan.

Rentang memakai min–median–max jika sampel <4, lalu kuartil 25–median–kuartil 75 untuk sampel lebih banyak. Rentang ini **deskriptif**, bukan interval kepercayaan. Sisa layanan dikurangi waktu yang telah berjalan, dengan cadangan kecil bila sudah melampaui kisaran historis.


In [ ]:
from collections import deque
from copy import deepcopy
import math


def foot_point(box):
    x1, y1, x2, y2 = box
    return ((float(x1) + float(x2)) / 2, float(y2))


def signed_distance(pt, line):
    (x1, y1), (x2, y2) = line["p1"], line["p2"]
    length = math.hypot(x2 - x1, y2 - y1)
    if length <= 0:
        raise ValueError("p1 dan p2 garis harus berbeda")
    return ((x2 - x1) * (pt[1] - y1) - (y2 - y1) * (pt[0] - x1)) / length


def side_of_line(pt, line, deadband=4.0):
    d = signed_distance(pt, line)
    return 0 if abs(d) <= deadband else (1 if d > 0 else -1)


def within_segment(pt, line, margin=0.0):
    (x1, y1), (x2, y2) = line["p1"], line["p2"]
    length = math.hypot(x2 - x1, y2 - y1)
    projection = ((pt[0] - x1) * (x2 - x1) + (pt[1] - y1) * (y2 - y1)) / length
    return -margin <= projection <= length + margin


def segment_crossing(a, b, line, margin=15.0):
    # Periksa TITIK POTONG lintasan kaki dengan ruas garis, bukan posisi akhir bbox.
    da, db = signed_distance(a, line), signed_distance(b, line)
    if da * db > 0 or abs(da - db) < 1e-10:
        return False
    t = da / (da - db)
    hit = (a[0] + t * (b[0] - a[0]), a[1] + t * (b[1] - a[1]))
    return 0 <= t <= 1 and within_segment(hit, line, margin)


def point_in_polygon(pt, polygon):
    if not polygon:
        return False
    x, y = pt
    inside = False
    for i, (x1, y1) in enumerate(polygon):
        x2, y2 = polygon[(i + 1) % len(polygon)]
        cross = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)
        if abs(cross) < 1e-7 and min(x1, x2) - 1e-7 <= x <= max(x1, x2) + 1e-7 and min(y1, y2) - 1e-7 <= y <= max(y1, y2) + 1e-7:
            return True
        if (y1 > y) != (y2 > y):
            if x < (x2 - x1) * (y - y1) / (y2 - y1) + x1:
                inside = not inside
    return inside


def scale_lanes(lanes, size, reference=(640, 360)):
    sx, sy = size[0] / reference[0], size[1] / reference[1]
    result = deepcopy(lanes)
    for cfg in result.values():
        for line in cfg["entry"] + [cfg["exit"]]:
            for key in ("p1", "p2", "after"):
                x, y = line[key]
                line[key] = (x * sx, y * sy)
        for key in ("queue_roi", "service_roi"):
            cfg[key] = [(x * sx, y * sy) for x, y in cfg.get(key, [])]
    return result


class QueueCounter:
    """Satu kasir, satu antrean FIFO. Jam internal memakai frame/FPS video."""
    def __init__(self, name, entry_lines, exit_line, fps, shared_samples=None,
                 deadband=4.0, lost_timeout_s=15.0, merge_s=3.0, merge_px=45.0,
                 min_samples=2, recent_n=12, queue_roi=None, service_roi=None,
                 allow_recovery=False, bootstrap_s=0.0, recovery_stable_s=0.8,
                 service_confirm_s=0.6, prior_range=None, use_shared=False,
                 segment_margin=15.0, max_cross_gap_s=6.0,
                 min_service_s=1.0, max_service_s=900.0,
                 exit_deadband=2.0, recovery_exit_guard=35.0,
                 outside_roi_timeout_s=3.0):
        if isinstance(entry_lines, dict):
            entry_lines = [entry_lines]
        if len(entry_lines) != 1:
            raise ValueError("Notebook ini memakai tepat satu garis masuk per kasir")
        if not math.isfinite(float(fps)) or fps <= 0:
            raise ValueError("FPS harus positif")
        self.name, self.fps = name, float(fps)
        self.lines = {"entry0": deepcopy(entry_lines[0]), "exit": deepcopy(exit_line)}
        self.entry_keys = ["entry0"]
        self.after = {k: side_of_line(v["after"], v, 0) for k, v in self.lines.items()}
        if 0 in self.after.values():
            raise ValueError("after harus berada di salah satu sisi garis")
        self.deadband, self.segment_margin = float(deadband), float(segment_margin)
        self.exit_deadband = float(exit_deadband)
        self.recovery_exit_guard = float(recovery_exit_guard)
        self.outside_roi_timeout_s = float(outside_roi_timeout_s)
        self.max_cross_gap = max_cross_gap_s * self.fps
        self.lost_timeout, self.merge_frames, self.merge_px = lost_timeout_s, merge_s * self.fps, merge_px
        self.min_samples, self.recent_n = min_samples, recent_n
        self.queue_roi, self.service_roi = queue_roi, service_roi
        self.allow_recovery, self.bootstrap_s = allow_recovery, bootstrap_s
        self.recovery_stable_s, self.service_confirm_s = recovery_stable_s, service_confirm_s
        self.prior_range = tuple(map(float, prior_range)) if prior_range is not None else None
        if self.prior_range is not None and (len(self.prior_range) != 3 or not 0 < self.prior_range[0] <= self.prior_range[1] <= self.prior_range[2]):
            raise ValueError("prior_range harus (bawah, median, atas), positif dan berurutan")
        self.use_shared = use_shared
        self.min_service_s, self.max_service_s = min_service_s, max_service_s
        self.shared = shared_samples if shared_samples is not None else []
        self.alias, self.seen_raw = {}, set()
        self.last_side = {key: {} for key in self.lines}
        self.last_seen, self.last_pos, self.last_box, self.observations = {}, {}, {}, {}
        self.in_queue, self.records = {}, {}
        self.done, self.done_state, self.ignored_exits = set(), {}, set()
        self.entries = self.exits = self.lost = self.merged = 0
        self.initial = self.recovered = 0
        self.service_samples, self.wait_times, self.events = [], [], []
        self.serving_id = None
        self.last_frame = -1
        self.last_exit_frame = None
        self.backlog_after_last_exit = False

    def _log(self, f, event, tid, **kw):
        self.events.append({"lane": self.name, "frame": int(f), "time_s": f / self.fps,
                            "event": event, "track_id": int(tid), "queue": self.queue_len(), **kw})

    def _crossed(self, key, tid, pt, f):
        band = self.exit_deadband if key == "exit" else self.deadband
        side = side_of_line(pt, self.lines[key], band)
        if side == 0:
            return False  # Simpan titik stabil SEBELUM pita mati, jangan timpa dengan jitter.
        previous = self.last_side[key].get(tid)
        self.last_side[key][tid] = (side, pt, f)
        return bool(previous is not None and previous[0] == -self.after[key]
                    and side == self.after[key] and f - previous[2] <= self.max_cross_gap
                    and segment_crossing(previous[1], pt, self.lines[key], self.segment_margin))

    def _find_lost_person(self, f, pt, box, visible):
        candidates = []
        area = max(1.0, (box[2] - box[0]) * (box[3] - box[1]))
        for tid in self.in_queue:
            gap = f - self.last_seen.get(tid, f)
            if tid in visible or not 0 < gap <= self.merge_frames:
                continue
            old = self.last_box[tid]
            old_area = max(1.0, (old[2] - old[0]) * (old[3] - old[1]))
            distance = math.dist(pt, self.last_pos[tid])
            if distance <= self.merge_px and 0.4 <= area / old_area <= 2.5:
                candidates.append((distance, tid))
        candidates.sort()
        if not candidates or (len(candidates) > 1 and candidates[1][0] - candidates[0][0] < 10):
            return None  # Jangan menebak bila dua orang hilang sama-sama dekat.
        return candidates[0][1]

    def _lose(self, tid, f, reason):
        self.in_queue.pop(tid)
        self.records[tid]["lost_frame"] = f
        self.done.add(tid)
        self.done_state[tid] = "LOST"
        self.lost += 1
        if self.serving_id == tid:
            self.serving_id = None
        self._log(f, "LOST", tid, reason=reason)

    def _expire(self, f):
        for tid in list(self.in_queue):
            if (f - self.last_seen[tid]) / self.fps > self.lost_timeout:
                self._lose(tid, f, "NOT_VISIBLE")
        if not self.in_queue:
            self.backlog_after_last_exit = False

    def _expire_outside(self, f):
        if not self.queue_roi or self.outside_roi_timeout_s <= 0:
            return
        for tid in list(self.in_queue):
            since = self.records[tid].get("outside_since")
            if since is not None and (f - since) / self.fps > self.outside_roi_timeout_s:
                self._lose(tid, f, "OUTSIDE_CUSTOMER_ZONE")
        if not self.in_queue:
            self.backlog_after_last_exit = False

    def _enter(self, tid, f, source):
        if tid in self.in_queue or tid in self.done:
            return
        obs = self.observations[tid]
        estimate = self.estimate()
        self.records[tid] = {
            "entry_frame": f, "observed_frame": obs["first_frame"], "entry_source": source,
            "entry_known": source == "LINE", "ahead": self.queue_len(),
            "est_at_entry": estimate, "service_start_frame": None,
            "outside_since": None,
            "service_partial": source != "LINE" and obs["first_in_service"],
        }
        self.in_queue[tid] = f
        if source == "INITIAL":
            self.initial += 1
            event = "INITIAL"
            # Orang di depan saat video mulai harus tetap mendahului pendatang baru.
            ordered = sorted(self.in_queue, key=lambda q: signed_distance(
                self.last_pos.get(q, self.observations[q]["first_pos"]), self.lines["entry0"]
            ) * self.after["entry0"], reverse=True)
            self.in_queue = {q: self.in_queue[q] for q in ordered}
            if self.serving_id is not None and self.serving_id != ordered[0]:
                self.records[self.serving_id]["service_start_frame"] = None
                self.serving_id = None
        else:
            self.entries += 1
            self.recovered += int(source == "RECOVERED")
            event = "ENTRY" if source == "LINE" else "ENTRY_RECOVERED"
        self._log(f, event, tid, source=source, entry_time_known=source == "LINE")

    def _complete(self, tid, f):
        record = self.records[tid]
        entry_frame = self.in_queue.pop(tid)
        observed_total = (f - (entry_frame if record["entry_known"] else record["observed_frame"])) / self.fps
        service_start = record["service_start_frame"]
        service_s, sample, sample_source = None, None, None
        if service_start is not None:
            service_s = (f - service_start) / self.fps
            if not record["service_partial"] and self.min_service_s <= service_s <= self.max_service_s:
                sample, sample_source = service_s, "SERVICE_ZONE"
        if sample is None and self.last_exit_frame is not None and self.backlog_after_last_exit:
            cycle = (f - self.last_exit_frame) / self.fps
            if self.min_service_s <= cycle <= self.max_service_s:
                sample, sample_source = cycle, "BUSY_EXIT_INTERVAL"
        if sample is not None:
            self.service_samples.append(float(sample))
            self.shared.append(float(sample))
        self.exits += 1
        self.wait_times.append(observed_total)
        record.update(exit_frame=f, total_observed_s=observed_total, service_s=service_s,
                      service_sample_s=sample, sample_source=sample_source)
        self.done.add(tid)
        self.done_state[tid] = "DONE"
        if self.serving_id == tid:
            self.serving_id = None
        self.last_exit_frame = f
        self.backlog_after_last_exit = self.queue_len() > 0
        est = record["est_at_entry"] or (np.nan,) * 3
        self._log(f, "EXIT", tid, wait_s=observed_total,
                  entry_time_known=record["entry_known"], source=record["entry_source"],
                  service_s=service_s, service_sample_s=sample, sample_source=sample_source,
                  est_lo=est[0], est_mid=est[1], est_hi=est[2])

    def _start_front_service(self, f, visible):
        if self.serving_id is not None or not self.in_queue:
            return
        tid = next(iter(self.in_queue))
        if tid not in visible:
            return
        obs = self.observations[tid]
        if self.service_roi:
            since = obs["service_since"]
            if since is None or (f - since) / self.fps < self.service_confirm_s:
                return
        record = self.records[tid]
        if record["service_start_frame"] is None:
            record["service_start_frame"] = f
            self._log(f, "SERVICE_START", tid, partial=record["service_partial"])
        self.serving_id = tid

    def update(self, f, tracks):
        f = int(f)
        if f <= self.last_frame:
            raise ValueError("update harus sekali per frame, dengan frame yang terus meningkat")
        self.last_frame = f
        start_event = len(self.events)
        tracks = [(int(t), list(map(float, b))) for t, b in tracks
                  if len(b) == 4 and all(math.isfinite(float(v)) for v in b)]
        visible = {self.alias.get(t, t) for t, _ in tracks}
        # Track yang benar-benar terlihat sekarang tidak kedaluwarsa hanya akibat jarak frame update.
        for tid in visible:
            if tid in self.in_queue:
                self.last_seen[tid] = f
        self._expire(f)
        # Pada frame awal, pelanggan paling depan didaftarkan dahulu agar FIFO benar.
        tracks.sort(key=lambda item: signed_distance(foot_point(item[1]), self.lines["entry0"]) * self.after["entry0"], reverse=True)
        for raw, box in tracks:
            pt = foot_point(box)
            if raw not in self.seen_raw:
                dup = self._find_lost_person(f, pt, box, visible)
                self.seen_raw.add(raw)
                if dup is not None:
                    self.alias[raw] = dup
                    visible.add(dup)
                    self.merged += 1
                    self._log(f, "ID_MERGED", raw, merged_into=dup)
            tid = self.alias.get(raw, raw)
            visible.add(tid)
            inside = point_in_polygon(pt, self.queue_roi)
            in_service = point_in_polygon(pt, self.service_roi)
            if tid not in self.observations:
                self.observations[tid] = {"first_frame": f, "first_pos": pt,
                    "first_in_service": in_service, "inside_since": f if inside else None,
                    "service_since": f if in_service else None, "last_frame": f}
            obs = self.observations[tid]
            gap = f - obs["last_frame"]
            if not inside:
                obs["inside_since"] = None
            elif obs["inside_since"] is None or gap > self.fps:
                obs["inside_since"] = f
            if not in_service:
                obs["service_since"] = None
            elif obs["service_since"] is None or gap > self.fps:
                obs["service_since"] = f
            obs["last_frame"] = f
            crossed_in = self._crossed("entry0", tid, pt, f)
            crossed_out = self._crossed("exit", tid, pt, f)
            if crossed_in and (self.queue_roi is None or inside):
                self._enter(tid, f, "LINE")
            if self.allow_recovery and inside and tid not in self.in_queue and tid not in self.done:
                entry_dist = signed_distance(pt, self.lines["entry0"]) * self.after["entry0"]
                exit_dist = signed_distance(pt, self.lines["exit"]) * self.after["exit"]
                stable = obs["inside_since"] is not None and (f - obs["inside_since"]) / self.fps >= self.recovery_stable_s
                initial = self.bootstrap_s > 0 and obs["first_frame"] / self.fps <= self.bootstrap_s
                moving_up = (signed_distance(pt, self.lines["entry0"]) - signed_distance(obs["first_pos"], self.lines["entry0"])) * self.after["entry0"] < -self.deadband
                # ID yang baru muncul dekat exit tidak dianggap entry baru. ID antrean
                # yang sudah ada tetap bisa keluar, termasuk sesudah ID_MERGED.
                safely_before_exit = exit_dist < -max(self.deadband, self.recovery_exit_guard)
                if stable and entry_dist > self.deadband and safely_before_exit and (initial or not moving_up):
                    self._enter(tid, f, "INITIAL" if initial else "RECOVERED")
            self.last_seen[tid], self.last_pos[tid], self.last_box[tid] = f, pt, box
            if tid in self.in_queue and self.queue_roi:
                # Jalur mendekati exit tetap sah meski melewati tepi ROI.
                approaching_exit = (within_segment(pt, self.lines["exit"], self.segment_margin)
                    and abs(signed_distance(pt, self.lines["exit"])) <= max(2 * self.recovery_exit_guard, self.exit_deadband))
                record = self.records[tid]
                if inside or approaching_exit:
                    record["outside_since"] = None
                elif record["outside_since"] is None:
                    record["outside_since"] = f
            if crossed_out:
                if tid in self.in_queue:
                    self._complete(tid, f)
                elif tid not in self.done and tid not in self.ignored_exits:
                    self.ignored_exits.add(tid)
                    self._log(f, "EXIT_IGNORED_NOT_REGISTERED", tid)
        self._expire_outside(f)
        self._start_front_service(f, visible)
        assert self.queue_len() == self.entries + self.initial - self.exits - self.lost
        return [e["event"] for e in self.events[start_event:]]

    def queue_len(self):
        return len(self.in_queue)

    def waiting_len(self):
        return max(0, self.queue_len() - int(self.serving_id is not None))

    def per_person_range(self):
        samples = self.service_samples
        if len(samples) < self.min_samples and self.use_shared:
            samples = self.shared
        if len(samples) < self.min_samples:
            return self.prior_range
        samples = np.asarray(samples[-self.recent_n:], dtype=float)
        lo, hi = (np.percentile(samples, 25), np.percentile(samples, 75)) if len(samples) >= 4 else (samples.min(), samples.max())
        return float(lo), float(np.median(samples)), float(hi)

    def estimate_source(self):
        if len(self.service_samples) >= self.min_samples:
            return f"data kasir ({len(self.service_samples)} sampel)"
        if self.use_shared and len(self.shared) >= self.min_samples:
            return f"data gabungan ({len(self.shared)} sampel)"
        return "asumsi awal/manual" if self.prior_range is not None else "kalibrasi"

    def _remaining_service(self, service_range):
        if self.serving_id is None:
            return np.zeros(3, dtype=float)
        start = self.records[self.serving_id]["service_start_frame"]
        elapsed = max(0.0, (self.last_frame - start) / self.fps)
        # Sisa waktu tidak dibuat negatif; bila layanan melampaui kisaran, sisakan cadangan kecil.
        reserve = min(5.0, 0.1 * float(service_range[1]))
        return np.maximum(np.asarray(service_range) - elapsed, reserve)

    def wait_to_start(self):
        service = self.per_person_range()
        if service is None:
            return None
        result = self._remaining_service(service) + self.waiting_len() * np.asarray(service)
        return tuple(map(float, result))

    def estimate(self):
        service = self.per_person_range()
        if service is None:
            return None
        return tuple(map(float, np.asarray(self.wait_to_start()) + np.asarray(service)))

    def state_of(self, tid):
        tid = self.alias.get(int(tid), int(tid))
        return "SERVING" if tid == self.serving_id else ("QUEUE" if tid in self.in_queue else self.done_state.get(tid, "OUT"))

    def elapsed(self, tid):
        tid = self.alias.get(int(tid), int(tid))
        if tid not in self.in_queue:
            return None
        record = self.records[tid]
        start = record["entry_frame"] if record["entry_known"] else record["observed_frame"]
        return max(0.0, (self.last_frame - start) / self.fps)


def fmt_time(seconds):
    if seconds is None or not math.isfinite(float(seconds)):
        return "kalibrasi"
    seconds = max(0, int(round(seconds)))
    minutes, sec = divmod(seconds, 60)
    return f"{minutes} m {sec:02d} dtk" if minutes else f"{sec} dtk"


def fmt_clock(seconds):
    seconds = max(0, int(round(seconds)))
    return f"{seconds // 60:02d}:{seconds % 60:02d}"


def fmt_range(est):
    return "kalibrasi" if est is None else f"{fmt_time(est[0])} - {fmt_time(est[2])}"


## 4. Tes logika otomatis — tanpa video/model
Tes diturunkan dari konfigurasi `LANES`, dijalankan untuk **kedua kasir**, dan mencakup crossing miring, arah terbalik, jitter, duplikasi, isolasi lane, timeout, ID putus pada exit, antrean awal, pelanggan terlambat terdeteksi, petugas di luar zona, FIFO awal, perhitungan waktu, exit samping, dan pencegahan entry palsu dekat exit.

Hasil yang sudah tersimpan di bawah berasal dari pengujian logika revisi ini. Hasil tersebut **bukan pengukuran akurasi YOLO** pada video. Jalankan ulang cell ini ketika konfigurasi diubah.


In [ ]:
# Tes ini tidak membutuhkan video, model YOLO, atau GPU.
TEST_ROWS = []

def test_check(lane, name, condition):
    TEST_ROWS.append({"lane": lane, "test": name, "hasil": "PASS" if bool(condition) else "FAIL"})

def test_point(line, distance=0, t=0.5):
    a, b = np.asarray(line["p1"], float), np.asarray(line["p2"], float)
    direction = b - a
    normal = np.array([-direction[1], direction[0]]) / np.linalg.norm(direction)
    normal *= side_of_line(line["after"], line, 0)
    return tuple(a + t * direction + distance * normal)

def test_detection(tid, point):
    x, y = point
    return tid, [x - 10, y - 80, x + 10, y]

def test_walk(qc, tid, points, start=0):
    for i, p in enumerate(points):
        qc.update(start + i, [test_detection(tid, p)])
    return start + len(points) - 1

def test_route(cfg):
    e, x = cfg["entry"][0], cfg["exit"]
    return [test_point(e, -20)] + list(np.linspace(test_point(e, 20), test_point(x, -20), 20)) + [test_point(x, 20)]

def test_counter(name, **kw):
    cfg = LANES[name]
    return QueueCounter(name, cfg["entry"], cfg["exit"], fps=10, **kw)

def test_counter_cfg(name, **kw):
    # Counter dengan margin/pita mati dari konfigurasi notebook (SEGMENT_MARGIN_PX, EXIT_DEADBAND_PX).
    return test_counter(name, segment_margin=SEGMENT_MARGIN_PX, exit_deadband=EXIT_DEADBAND_PX, **kw)

def test_serve(qc, cfg, tid, start, duration):
    e, x = cfg["entry"][0], cfg["exit"]
    steps = int(round(duration * qc.fps))
    pts = [test_point(e, -20)] + list(np.linspace(test_point(e, 20), test_point(x, -20), steps)) + [test_point(x, 20)]
    test_walk(qc, tid, pts, start=start - 1)

assert set(LANES) == {"Kasir Tengah", "Kasir Kanan"}
assert all(len(c["entry"]) == 1 for c in LANES.values())

for name, cfg in LANES.items():
    entry, exit_line = cfg["entry"][0], cfg["exit"]
    before, after = test_point(entry, -20), test_point(entry, 20)
    route = test_route(cfg)

    q = test_counter(name)
    test_walk(q, 1, route)
    test_check(name, "01 lintasan penuh: masuk 1, keluar 1, antrean 0", (q.entries, q.exits, q.queue_len()) == (1, 1, 0))

    q = test_counter(name)
    test_walk(q, 2, [after, before])
    test_check(name, "02 arah masuk terbalik tidak dihitung", (q.entries, q.exits) == (0, 0))

    q = test_counter(name)
    test_walk(q, 3, [test_point(entry, d) for d in (-2, 1, -1, 2) * 4])
    test_check(name, "03 jitter dalam pita mati tidak dihitung", q.entries == 0)

    q = test_counter(name)
    test_walk(q, 4, [before, after] * 4)
    test_check(name, "04 entry bolak-balik tidak dihitung ganda", q.entries == 1)

    q = test_counter(name)
    pts = route + [test_point(exit_line, -20), test_point(exit_line, 20)] * 3
    test_walk(q, 5, pts)
    test_check(name, "05 exit bolak-balik tidak dihitung ganda", q.exits == 1)

    q = test_counter(name)
    test_walk(q, 6, [(50, 5), (50, 340)])
    test_check(name, "06 di luar ruas garis tidak dihitung", (q.entries, q.exits) == (0, 0))

    q = test_counter(name)
    other = next(c for n, c in LANES.items() if n != name)
    test_walk(q, 7, test_route(other))
    test_check(name, "07 lintasan kasir lain tidak dihitung", (q.entries, q.exits) == (0, 0))

    q = test_counter(name, lost_timeout_s=5)
    last = test_walk(q, 8, [before, after])
    q.update(last + 30, [])
    retained = q.queue_len() == 1
    q.update(last + 60, [])
    test_check(name, "08 hilang 3 s ditahan, >5 s menjadi LOST", retained and (q.lost, q.queue_len()) == (1, 0))

    q = test_counter(name)
    test_walk(q, 9, route[:-1])
    q.update(len(route) - 1, [])
    # ID baru muncul SETELAH exit; sisi stabil ID lama harus tetap tersimpan.
    q.update(len(route), [test_detection(90, test_point(exit_line, 20))])
    test_check(name, "09 ID putus di exit tetap tersambung dan selesai", (q.entries, q.exits, q.merged) == (1, 1, 1))

    q = test_counter(name)
    for f, d in enumerate((-20, 20)):
        q.update(f, [test_detection(10, test_point(entry, d, 0.3)), test_detection(11, test_point(entry, d, 0.7))])
    test_check(name, "10 dua orang terlihat bersamaan tidak digabung", (q.entries, q.queue_len(), q.merged) == (2, 2, 0))

    q = test_counter(name)
    test_walk(q, 12, [test_point(exit_line, -20), test_point(exit_line, 20)])
    test_check(name, "11 exit tanpa registrasi dicatat terpisah", q.exits == 0 and any(e["event"] == "EXIT_IGNORED_NOT_REGISTERED" for e in q.events))

    q = test_counter(name, min_samples=2)
    test_check(name, "12 tanpa sampel dan tanpa prior: kalibrasi", q.estimate() is None)
    for i, duration in enumerate((10, 20, 30)):
        test_serve(q, cfg, 100 + i, 1000 * (i + 1), duration)
    for f, d in ((5000, -20), (5001, 20)):
        q.update(f, [test_detection(200, test_point(entry, d, 0.3)), test_detection(201, test_point(entry, d, 0.7))])
    est = q.estimate()
    test_check(name, "13 2 orang + layanan 10/20/30 s: selesai 30/60/90 s", q.queue_len() == 2 and est is not None and np.allclose(est, (30, 60, 90)))

    # Regresi: titik akhir di luar ujung ruas, tetapi titik potong sebenarnya di dalam.
    special = {"p1": (100, 100), "p2": (120, 100), "after": (110, 130)}
    test_check(name, "14 titik potong lintasan diperiksa, bukan titik akhir", segment_crossing((90, 80), (130, 120), special, margin=0))

    center = tuple(np.mean(np.asarray(cfg["service_roi"]), axis=0))
    q = test_counter(name, queue_roi=cfg["queue_roi"], service_roi=cfg["service_roi"], allow_recovery=True, bootstrap_s=2)
    test_walk(q, 300, [center] * 12)
    test_check(name, "15 pelanggan awal terdaftar INITIAL sekali", (q.initial, q.entries, q.queue_len()) == (1, 0, 1))
    test_walk(q, 300, list(np.linspace(center, test_point(exit_line, -20), 10)) + [test_point(exit_line, 20)], start=12)
    test_check(name, "16 pelanggan awal bisa EXIT tanpa entry palsu", (q.initial, q.entries, q.exits, q.queue_len()) == (1, 0, 1, 0))
    test_check(name, "17 layanan awal yang terpotong bukan sampel penuh", len(q.service_samples) == 0)

    q = test_counter(name, queue_roi=cfg["queue_roi"], service_roi=cfg["service_roi"], allow_recovery=True, bootstrap_s=2)
    test_walk(q, 301, [center] * 12, start=100)
    test_check(name, "18 baru terdeteksi setelah entry: RECOVERED sekali", (q.recovered, q.entries, q.queue_len()) == (1, 1, 1))

    q = test_counter(name, queue_roi=cfg["queue_roi"], service_roi=cfg["service_roi"], allow_recovery=True, bootstrap_s=2)
    test_walk(q, 302, [(296, 166)] * 15)
    test_walk(q, 303, [(471, 180)] * 15, start=15)
    test_check(name, "19 titik kaki petugas di luar zona pelanggan diabaikan", q.queue_len() == 0 and q.initial == 0 and q.entries == 0)

    q = test_counter(name, prior_range=(25, 45, 90))
    test_check(name, "20 prior langsung tampil, berlabel asumsi awal", np.allclose(q.estimate(), (25, 45, 90)) and q.estimate_source() == "asumsi awal/manual")
    for f in range(102):
        d = -20 if f == 0 else 20
        q.update(f, [test_detection(400, test_point(entry, d, 0.3)), test_detection(401, test_point(entry, d, 0.7))])
    test_check(name, "21 estimasi mengurangi waktu layanan yang sudah berjalan", np.allclose(q.estimate(), (65, 125, 260)))

    # Dua pelanggan antre bersamaan. Layanan kedua 20 s, bukan total waktunya dibagi panjang antrean.
    q = test_counter(name)
    for f in range(302):
        rows = []
        if f <= 101:
            p = before if f == 0 else (test_point(exit_line, -20) if f <= 100 else test_point(exit_line, 20))
            if f == 1:
                p = after
            rows.append(test_detection(500, p))
        if f >= 4:
            p = before if f == 4 else (test_point(exit_line, -20) if f <= 300 else test_point(exit_line, 20))
            if f == 5:
                p = after
            rows.append(test_detection(501, p))
        q.update(f, rows)
    test_check(name, "22 sampel layanan dua pelanggan: 10 s dan 20 s", q.exits == 2 and np.allclose(q.service_samples, (10, 20)))

    q = test_counter(name, queue_roi=cfg["queue_roi"], service_roi=cfg["service_roi"], allow_recovery=True, bootstrap_s=2)
    for f in range(12):
        rows = [test_detection(600, center)]
        if f >= 5:
            rows.append(test_detection(601, before if f == 5 else after))
        q.update(f, rows)
    test_check(name, "23 pelanggan awal tetap di depan pendatang baru", q.queue_len() == 2 and next(iter(q.in_queue)) == 600)

    # Regresi dari review video output: pelanggan dapat lewat di kiri ruas exit lama.
    old_exit = ({"p1": (370, 304), "p2": (450, 299), "after": (410, 335)}
                if name == "Kasir Tengah" else
                {"p1": (612, 271), "p2": (637, 264), "after": (620, 305)})
    lateral_before = test_point(exit_line, -20, t=0.2)
    lateral_after = test_point(exit_line, 20, t=0.2)
    lateral_path = [before, after] + list(np.linspace(after, lateral_before, 15)) + [lateral_after]
    q = test_counter(name)
    old_q = QueueCounter(name, cfg["entry"], old_exit, fps=10)
    test_walk(q, 700, lateral_path)
    test_walk(old_q, 700, lateral_path)
    test_check(name, "24 exit samping yang lolos garis lama kini dihitung", (q.exits, old_q.exits) == (1, 0))

    q = test_counter(name)
    test_walk(q, 701, [before, after, test_point(exit_line, -20, t=-0.5), test_point(exit_line, 20, t=-0.5)])
    test_check(name, "25 exit tetap terbatas ruas, tidak melebar ke kasir lain", q.exits == 0)

    q = test_counter(name)
    test_walk(q, 702, [before, after, test_point(exit_line, -15)] + [test_point(exit_line, d) for d in (-1, 1, -1.5, 1.5) * 3])
    no_jitter_exit = q.exits == 0
    q.update(q.last_frame + 1, [test_detection(702, test_point(exit_line, 5))])
    test_check(name, "26 jitter exit diabaikan, crossing stabil dihitung sekali", no_jitter_exit and q.exits == 1)

    q = test_counter(name, queue_roi=cfg["queue_roi"], service_roi=cfg["service_roi"], allow_recovery=True, bootstrap_s=2)
    near_exit = test_point(exit_line, -10)
    assert point_in_polygon(near_exit, cfg["queue_roi"]), "Titik regresi harus berada dalam ROI"
    last = test_walk(q, 703, [near_exit] * 12, start=100)
    q.update(last + 1, [test_detection(703, test_point(exit_line, 20))])
    test_check(name, "27 ID baru dekat exit tidak menjadi entry/exit palsu", (q.entries, q.recovered, q.exits, q.queue_len()) == (0, 0, 0, 0))

    q = test_counter(name, queue_roi=cfg["queue_roi"], service_roi=cfg["service_roi"], allow_recovery=True, bootstrap_s=2)
    test_walk(q, 704, route)
    test_check(name, "28 zona larangan entry tidak menghalangi exit pelanggan aktif", (q.entries, q.exits, q.queue_len()) == (1, 1, 0))

    q = test_counter(name, queue_roi=cfg["queue_roi"], outside_roi_timeout_s=3)
    test_walk(q, 705, [before, after] + [(296, 166)] * 42)
    test_check(name, "29 ID menetap di area petugas menjadi LOST, bukan EXIT", (q.entries, q.exits, q.lost, q.queue_len()) == (1, 0, 1, 0)
               and any(e.get("reason") == "OUTSIDE_CUSTOMER_ZONE" for e in q.events))

    q = test_counter(name, queue_roi=cfg["queue_roi"], outside_roi_timeout_s=3)
    test_walk(q, 706, [before, after] + [lateral_before] * 42 + [lateral_after])
    test_check(name, "30 mendekati tepi exit tetap aktif dan keluar sah", (q.entries, q.exits, q.lost, q.queue_len()) == (1, 1, 0, 0))

    # ---- Regresi revisi garis EXIT ----
    ends = (exit_line["p1"], exit_line["p2"])
    test_check(name, "31 kedua ujung garis exit berada di dalam queue_roi",
               all(point_in_polygon(p, cfg["queue_roi"]) for p in ends))

    length = math.dist(*ends)
    ok_edges = True
    for t_edge in (0.03, 0.97):                                  # lewat tepat di pinggir ruas kiri / kanan
        q = test_counter_cfg(name)
        test_walk(q, 800, [before, after, test_point(exit_line, -15, t_edge), test_point(exit_line, 15, t_edge)])
        ok_edges &= (q.exits == 1)
    test_check(name, "32 pelanggan yang lewat di pinggir kiri/kanan exit tetap terhitung", ok_edges)

    q = test_counter_cfg(name)
    t_in = 1 + 0.8 * SEGMENT_MARGIN_PX / length                  # sedikit di luar ujung, masih dalam margin
    test_walk(q, 801, [before, after, test_point(exit_line, -15, t_in), test_point(exit_line, 15, t_in)])
    q2 = test_counter_cfg(name)
    t_out = 1 + 2.0 * SEGMENT_MARGIN_PX / length                 # jauh di luar margin -> bukan exit
    test_walk(q2, 802, [before, after, test_point(exit_line, -15, t_out), test_point(exit_line, 15, t_out)])
    test_check(name, "33 margin ujung: dalam margin terhitung, jauh di luar margin tidak", (q.exits, q2.exits) == (1, 0))

    a_dist = signed_distance(exit_line["after"], exit_line) * side_of_line(exit_line["after"], exit_line, 0)
    test_check(name, "34 vektor after berada di sisi luar (bawah) garis exit, jarak >= 25 px",
               a_dist >= 25 and exit_line["after"][1] > max(exit_line["p1"][1], exit_line["p2"][1]))

TEST_DF = pd.DataFrame(TEST_ROWS)
print(TEST_DF.to_string(index=False))
failed = TEST_DF[TEST_DF["hasil"] == "FAIL"]
assert failed.empty, "Tes gagal:\n" + failed.to_string(index=False)
print(f"\nLULUS: {len(TEST_DF)}/{len(TEST_DF)} tes logika dan geometri. Belum merupakan akurasi video nyata.")


        lane                                                                   test hasil
Kasir Tengah                        01 lintasan penuh: masuk 1, keluar 1, antrean 0  PASS
Kasir Tengah                                  02 arah masuk terbalik tidak dihitung  PASS
Kasir Tengah                               03 jitter dalam pita mati tidak dihitung  PASS
Kasir Tengah                              04 entry bolak-balik tidak dihitung ganda  PASS
Kasir Tengah                               05 exit bolak-balik tidak dihitung ganda  PASS
Kasir Tengah                                   06 di luar ruas garis tidak dihitung  PASS
Kasir Tengah                                  07 lintasan kasir lain tidak dihitung  PASS
Kasir Tengah                               08 hilang 3 s ditahan, >5 s menjadi LOST  PASS
Kasir Tengah                       09 ID putus di exit tetap tersambung dan selesai  PASS
Kasir Tengah                         10 dua orang terlihat bersamaan tidak digabung  PASS
Kasir Teng

## 5. Membuka video dan membaca jam video
Jika path default belum tersedia, notebook mencoba menemukan satu file video di folder kerja/Colab/Kaggle. Di Colab, bila belum ditemukan, dialog unggah akan muncul. Gunakan rekaman asli, bukan output yang sudah berisi kotak/teks.

Waktu peristiwa dan estimasi memakai **nomor frame/FPS rekaman**. Komputer yang memproses lebih lambat tidak membuat durasi antrean video menjadi lebih lama.


In [ ]:
def resolve_video(path):
    if isinstance(path, int):
        return path
    path = Path(path)
    if path.exists():
        return str(path)
    extensions = {".mp4", ".avi", ".mov", ".mkv", ".m4v", ".webm"}
    candidates = []
    roots = [Path.cwd(), Path("/content"), Path("/kaggle/input")]
    for root in roots:
        if root.exists():
            pattern = "**/*" if root == Path("/kaggle/input") else "*"
            candidates.extend(p for p in root.glob(pattern) if p.is_file() and p.suffix.lower() in extensions and p.parent.resolve() != OUT.resolve())
    candidates = list(dict.fromkeys(str(p.resolve()) for p in candidates))
    if len(candidates) == 1:
        print("Video ditemukan:", candidates[0])
        return candidates[0]
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Atur VIDEO_PATH ke video asli. Kandidat: {candidates[:8]}")
    print("Unggah SATU video asli kasir, bukan video output yang sudah memiliki overlay.")
    uploaded = files.upload()
    selected = [name for name in uploaded if Path(name).suffix.lower() in extensions]
    if len(selected) != 1:
        raise ValueError("Unggah tepat satu video atau atur VIDEO_PATH secara eksplisit")
    return str(Path(selected[0]).resolve())


SRC = resolve_video(VIDEO_PATH)
if isinstance(SRC, int) and MAX_FRAMES is None:
    raise ValueError("Untuk webcam, isi MAX_FRAMES agar proses dapat berhenti otomatis")
cap = cv2.VideoCapture(SRC)
if not cap.isOpened():
    raise RuntimeError(f"Video tidak bisa dibuka: {SRC}")
try:
    FPS = float(cap.get(cv2.CAP_PROP_FPS))
    if not math.isfinite(FPS) or FPS <= 0:
        FPS = 30.0
    frame_count = float(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    N_FRAMES = int(frame_count) if math.isfinite(frame_count) and frame_count > 0 else 0
    ok, FIRST = cap.read()
finally:
    cap.release()
if not ok or FIRST is None:
    raise RuntimeError("Frame pertama tidak terbaca")
H, W = FIRST.shape[:2]
# Cegah output berpanel dibaca lagi sebagai video mentah dan diskala keliru.
reference_ratio = REFERENCE_SIZE[0] / REFERENCE_SIZE[1]
looks_like_output_panel = (W > PANEL_W and
    abs(((W - PANEL_W) / H) / reference_ratio - 1) < 0.03 and
    abs((W / H) / reference_ratio - 1) > 0.20)
if looks_like_output_panel:
    raise ValueError("Video ini tampaknya sudah memiliki panel QueueWise. Jalankan ulang memakai VIDEO ASLI tanpa kotak/garis/panel, bukan video hasil download.")
ACTIVE_LANES = scale_lanes(LANES, (W, H), REFERENCE_SIZE)
PIXEL_SCALE = math.sqrt((W / REFERENCE_SIZE[0]) * (H / REFERENCE_SIZE[1]))
print(f"Video {W}x{H}, {FPS:.2f} FPS, {N_FRAMES} frame, {N_FRAMES / FPS:.1f} detik")
if abs((W / H) / (REFERENCE_SIZE[0] / REFERENCE_SIZE[1]) - 1) > 0.05:
    print("Rasio gambar berbeda dari referensi: periksa posisi garis dan zona pada preview berikut.")


## 6. Preview garis dan zona pelanggan
Hijau = masuk, merah = keluar, cyan = zona pelanggan, ungu = zona layanan. Panah menunjukkan arah crossing yang diterima. Periksa posisi **kaki**, bukan kepala atau pusat badan. Kedua ruas keluar diletakkan dekat posisi akhir kaki pelanggan dan diperlebar untuk menangkap lintasan samping, dengan toleransi (`SEGMENT_MARGIN_PX`) terbatas pada ujungnya; kedua ujung garis berada di dalam zona cyan, dan garis tetap tidak berlaku di seluruh lebar gambar.

Panel dashboard ditempatkan di **samping** video sehingga tidak menutup garis masuk. `INITIAL` dan `ENTRY_RECOVERED` tetap memerlukan pelanggan berada stabil di zona yang sesuai.


In [ ]:
COLORS = {"OUT": (160, 160, 160), "QUEUE": (0, 165, 255), "SERVING": (255, 210, 0),
          "DONE": (60, 210, 70), "LOST": (100, 100, 230)}

def show(img, title=""):
    plt.figure(figsize=(15, 7))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

def draw_lines(img, lanes, scale=1, show_roi=True):
    P = lambda p: (int(round(p[0] * scale)), int(round(p[1] * scale)))
    thickness = max(1, int(round(1.5 * scale)))
    if show_roi:
        layer = img.copy()
        for cfg in lanes.values():
            poly = np.asarray([P(p) for p in cfg["queue_roi"]], np.int32)
            cv2.fillPoly(layer, [poly], (90, 90, 20))
        cv2.addWeighted(layer, 0.15, img, 0.85, 0, dst=img)
        for cfg in lanes.values():
            for key, color in (("queue_roi", (230, 200, 40)), ("service_roi", (240, 90, 180))):
                poly = np.asarray([P(p) for p in cfg[key]], np.int32)
                cv2.polylines(img, [poly], True, color, thickness)
    for name, cfg in lanes.items():
        short = name.replace("Kasir ", "").upper()
        for label, line, color in ((f"{short} MASUK", cfg["entry"][0], (40, 230, 100)),
                                   (f"{short} KELUAR", cfg["exit"], (50, 50, 255))):
            cv2.line(img, P(line["p1"]), P(line["p2"]), color, thickness)
            a, b = np.asarray(line["p1"]), np.asarray(line["p2"])
            middle = (a + b) / 2
            normal = np.array([-(b-a)[1], (b-a)[0]], dtype=float)
            normal *= side_of_line(line["after"], line, 0) / np.linalg.norm(normal)
            cv2.arrowedLine(img, P(middle), P(middle + normal * 15 * PIXEL_SCALE), color, thickness, tipLength=0.4)
            fs = max(0.32, 0.30 * scale * PIXEL_SCALE)
            text_w = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, fs, thickness)[0][0]
            x = max(3, min(P(line["p1"])[0], img.shape[1] - text_w - 4))
            y = max(16, P(line["p1"])[1] - 5)
            cv2.putText(img, label, (x, y), cv2.FONT_HERSHEY_SIMPLEX, fs, color, thickness, cv2.LINE_AA)
    return img

def person_state(counters, tid):
    for c in counters.values():
        state = c.state_of(tid)
        if state in ("QUEUE", "SERVING"):
            return state, c
    for c in counters.values():
        state = c.state_of(tid)
        if state in ("DONE", "LOST"):
            return state, c
    return "OUT", None

def render_frame(frame, tracks, counters, fps_proc=0, trails=None):
    scale = OUT_SCALE
    view = cv2.resize(frame, (W * scale, H * scale), interpolation=cv2.INTER_LINEAR)
    draw_lines(view, ACTIVE_LANES, scale, SHOW_ROI)
    for tid, box in tracks:
        state, counter = person_state(counters, tid)
        color = COLORS[state]
        x1, y1, x2, y2 = box
        a, b = (int(x1 * scale), int(y1 * scale)), (int(x2 * scale), int(y2 * scale))
        cv2.rectangle(view, a, b, color, 2)
        foot = foot_point(box)
        p = (int(foot[0] * scale), int(foot[1] * scale))
        cv2.circle(view, p, 4, color, -1)
        elapsed = counter.elapsed(tid) if counter is not None else None
        label = f"ID{tid} {state}" + (f" {fmt_clock(elapsed)}" if elapsed is not None else "")
        cv2.putText(view, label, (max(0, a[0]), max(16, a[1] - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        if SHOW_TRAILS and trails is not None:
            trails.setdefault(tid, deque(maxlen=35)).append(p)
            if len(trails[tid]) > 1:
                cv2.polylines(view, [np.asarray(trails[tid], np.int32)], False, color, 1)
    # Panel diletakkan di samping video supaya tidak menutup garis masuk.
    panel = np.full((view.shape[0], PANEL_W, 3), (28, 22, 17), dtype=np.uint8)
    rows = [("QUEUEWISE", (90, 225, 230), 0.9),
            (f"Waktu video {fmt_clock(max(c.last_frame for c in counters.values()) / FPS)}", (200, 200, 200), 0.55)]
    estimates = []
    for name, c in counters.items():
        estimate = c.estimate()
        start_est = c.wait_to_start()
        rows.extend([
            ("", (255, 255, 255), 0.4),
            (name.upper(), (255, 255, 255), 0.7),
            (f"Menunggu {c.waiting_len()} | dilayani {int(c.serving_id is not None)} | total {c.queue_len()}", (225, 225, 225), 0.53),
            (f"Masuk {c.entries} | keluar {c.exits} | awal {c.initial}", (180, 190, 200), 0.52),
            ("JIKA BERGABUNG SEKARANG:", (90, 225, 230), 0.52),
            ("Mulai dilayani ~ " + (fmt_time(start_est[1]) if start_est is not None else "kalibrasi"), (230, 230, 230), 0.55),
            ("Sampai selesai ~ " + (fmt_time(estimate[1]) if estimate is not None else "kalibrasi"), (80, 235, 170), 0.62),
            ("Rentang: " + fmt_range(estimate), (210, 220, 220), 0.47),
            ("Sumber: " + c.estimate_source(), (160, 180, 200), 0.47),
        ])
        if estimate is not None:
            estimates.append((estimate[1], name, c.estimate_source()))
    rows.append(("", (255, 255, 255), 0.3))
    if estimates:
        _, fastest, source = min(estimates)
        rows.append(("Pilihan tercepat: " + fastest.replace("Kasir ", ""), (90, 225, 230), 0.55))
        rows.append(("Berdasar estimasi; bisa berubah.", (150, 170, 190), 0.45))
    rows.append((f"FPS proses {fps_proc:.1f} | video {FPS:.1f}", (150, 170, 190), 0.45))
    rows.append(("Zona pelanggan: cyan | layanan: ungu", (150, 170, 190), 0.43))
    line_h = max(10, min(29, (panel.shape[0] - 35) // max(1, len(rows))))
    for i, (text, color, requested_fs) in enumerate(rows):
        if not text:
            continue
        fs = min(requested_fs, line_h / 34)
        text_width = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, 1)[0][0]
        fs *= min(1.0, (PANEL_W - 26) / max(1, text_width))
        cv2.putText(panel, text, (14, 28 + i * line_h), cv2.FONT_HERSHEY_SIMPLEX, fs, color, 1, cv2.LINE_AA)
    return np.concatenate([view, panel], axis=1)

geo = draw_lines(cv2.resize(FIRST, (W * OUT_SCALE, H * OUT_SCALE)), ACTIVE_LANES, OUT_SCALE, True)
cv2.imwrite(str(OUT / "screenshots/01_garis_dan_zona.jpg"), geo)
show(geo, "Dua kasir: garis masuk/keluar + zona pelanggan dan layanan")
print("Periksa titik KAKI pelanggan berada di zona cyan; titik kaki petugas harus di luar zona.")


## 7. Uji YOLO pada beberapa waktu video
Model hanya mendeteksi kelas `person`. Preview menampilkan confidence, titik kaki, dan zona yang ditempati. Jika kaki petugas masuk zona pelanggan atau pelanggan berada di luar zona, koreksi polygon pada konfigurasi sebelum menilai hasil counting. Tidak ada pelatihan ulang.


In [ ]:
model_preview = YOLO(MODEL_NAME)
sample_frames = [0]
if N_FRAMES > 1 and not isinstance(SRC, int):
    sample_frames += [min(N_FRAMES - 1, int(30 * FPS)), min(N_FRAMES - 1, N_FRAMES // 2)]
cap = cv2.VideoCapture(SRC)
try:
    for frame_no in dict.fromkeys(sample_frames):
        if isinstance(SRC, int):
            frame = FIRST.copy()
        else:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
            ok, frame = cap.read()
            if not ok:
                continue
        r = model_preview.predict(frame, classes=[0], conf=CONF, iou=IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        img = frame.copy()
        draw_lines(img, ACTIVE_LANES, show_roi=True)
        for box, confidence in zip(r.boxes.xyxy.cpu().tolist(), r.boxes.conf.cpu().tolist()):
            x1, y1, x2, y2 = map(int, box)
            foot = foot_point(box)
            lane = next((n.replace("Kasir ", "") for n, cfg in ACTIVE_LANES.items() if point_in_polygon(foot, cfg["queue_roi"])), "luar zona")
            cv2.rectangle(img, (x1, y1), (x2, y2), (60, 230, 80), 1)
            cv2.circle(img, tuple(map(int, foot)), 4, (0, 0, 255), -1)
            cv2.putText(img, f"{confidence:.2f} {lane}", (max(0, x1), max(14, y1 - 4)), 0, 0.35, (60, 230, 80), 1, cv2.LINE_AA)
        cv2.imwrite(str(OUT / f"screenshots/02_deteksi_{frame_no:05d}.jpg"), img)
        show(img, f"Deteksi + titik kaki, t={frame_no / FPS:.1f} s: {len(r.boxes)} orang")
finally:
    cap.release()
del model_preview
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 8. Pipeline lengkap: tracking, dua antrean, timer, dan dashboard
Tracker diinisialisasi ulang sebelum memproses rekaman; `persist=True` menjaga ID antar-frame. Konfigurasi BoT-SORT disalin dari versi Ultralytics yang terpasang. Jika ReID tidak tersedia, pipeline mencoba ByteTrack.

- **QUEUE**: menunggu; **SERVING**: paling depan dan berada di area layanan; **DONE**: melewati garis keluar; **LOST**: terlalu lama tidak terlihat.
- Timer pada kotak adalah durasi teramati sejak masuk/pertama terlihat. Untuk `INITIAL`/`RECOVERED`, durasi sebelum terlihat tidak diketahui.
- **Mulai dilayani** dan **Sampai selesai** di panel berlaku untuk satu orang yang **baru akan bergabung** pada kondisi frame tersebut.
- **Sumber asumsi awal/manual** berarti estimasi belum berasal dari sampel aktual yang cukup. Rekomendasi kasir hanya membandingkan estimasi yang ditampilkan.
- Video berakhir tidak otomatis membuat orang yang masih menunggu dianggap keluar.


In [ ]:
import yaml
import ultralytics

def make_tracker_config(kind, with_reid=False):
    # Salin default dari versi yang terpasang agar field baru tetap tersedia.
    source = Path(ultralytics.__file__).resolve().parent / "cfg" / "trackers" / f"{kind}.yaml"
    cfg = yaml.safe_load(source.read_text())
    cfg.update(track_high_thresh=0.20, track_low_thresh=0.08, new_track_thresh=0.20,
               track_buffer=max(30, int(round(6 * FPS))), match_thresh=0.8, fuse_score=True)
    if kind == "botsort":
        cfg.update(gmc_method="none", with_reid=with_reid, model="auto",
                   proximity_thresh=0.5, appearance_thresh=0.25)
    target = OUT / f"{kind}_queuewise.yaml"
    target.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return str(target.resolve())

def make_model():
    tracker = make_tracker_config(TRACKER_TYPE, USE_REID)
    probe = YOLO(MODEL_NAME)
    try:
        probe.track(FIRST, persist=True, tracker=tracker, classes=[0], conf=CONF,
                    iou=IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)
    except Exception as error:
        if TRACKER_TYPE == "bytetrack":
            raise
        print(f"BoT-SORT/ReID tidak tersedia ({type(error).__name__}: {str(error)[:150]}). Mencoba ByteTrack.")
        tracker = make_tracker_config("bytetrack")
        probe = YOLO(MODEL_NAME)
        probe.track(FIRST, persist=True, tracker=tracker, classes=[0], conf=CONF,
                    iou=IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)
    del probe
    return YOLO(MODEL_NAME), tracker  # Model baru: frame 0 tidak diproses dua kali oleh tracker.

def make_counters():
    shared = []
    return {name: QueueCounter(
        name, cfg["entry"], cfg["exit"], FPS, shared_samples=shared,
        deadband=DEADBAND_PX * PIXEL_SCALE, segment_margin=SEGMENT_MARGIN_PX * PIXEL_SCALE,
        exit_deadband=EXIT_DEADBAND_PX * PIXEL_SCALE,
        recovery_exit_guard=RECOVERY_EXIT_GUARD_PX * PIXEL_SCALE,
        max_cross_gap_s=CROSSING_MAX_GAP_S, lost_timeout_s=LOST_TIMEOUT_S,
        outside_roi_timeout_s=OUTSIDE_ROI_TIMEOUT_S,
        merge_s=MERGE_S, merge_px=MERGE_PX * PIXEL_SCALE,
        min_samples=MIN_SAMPLES, recent_n=RECENT_N,
        queue_roi=cfg["queue_roi"], service_roi=cfg["service_roi"],
        allow_recovery=RECOVERY_ENABLED, bootstrap_s=BOOTSTRAP_S,
        recovery_stable_s=RECOVERY_STABLE_S, service_confirm_s=SERVICE_CONFIRM_S,
        prior_range=SERVICE_PRIOR_RANGE_S, use_shared=USE_SHARED_SAMPLES,
        min_service_s=MIN_SERVICE_SAMPLE_S, max_service_s=MAX_SERVICE_SAMPLE_S,
    ) for name, cfg in ACTIVE_LANES.items()}

model, TRACKER_USED = make_model()
counters = make_counters()
print("Model:", MODEL_NAME, "| device:", DEVICE, "| tracker:", Path(TRACKER_USED).name)
print("Sumber waktu awal:", "kalibrasi" if SERVICE_PRIOR_RANGE_S is None else f"asumsi manual {SERVICE_PRIOR_RANGE_S} detik/orang")
RAW = OUT / "queuewise_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v") if hasattr(cv2, "VideoWriter_fourcc") else cv2.VideoWriter.fourcc(*"mp4v")
output_size = (W * OUT_SCALE + PANEL_W, H * OUT_SCALE)
if any(d % 2 for d in output_size):
    raise ValueError("Dimensi video output harus genap: gunakan OUT_SCALE=2 dan PANEL_W genap")
writer = cv2.VideoWriter(str(RAW), fourcc, FPS, output_size)
if not writer.isOpened():
    raise RuntimeError("VideoWriter gagal dibuka; periksa codec MP4 dan folder output")
cap = cv2.VideoCapture(SRC)
if not cap.isOpened():
    writer.release()
    raise RuntimeError(f"Sumber video tidak bisa dibuka ulang: {SRC}")
timeline, track_log, latencies, snapshots, trails = [], [], [], {}, {}
f = 0
try:
    while MAX_FRAMES is None or f < MAX_FRAMES:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.perf_counter()
        result = model.track(frame, persist=True, tracker=TRACKER_USED, classes=[0], conf=CONF,
                             iou=IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        tracks = []
        if result.boxes is not None and result.boxes.id is not None:
            tracks = list(zip(result.boxes.id.int().cpu().tolist(), result.boxes.xyxy.cpu().tolist()))
        new_events = {name: c.update(f, tracks) for name, c in counters.items()}
        processing_fps = 1 / max(1e-6, np.mean(latencies[-30:])) if latencies else 0
        big = render_frame(frame, tracks, counters, processing_fps, trails)
        writer.write(big)
        # End-to-end: inferensi, logika, rendering dan penulisan frame.
        latencies.append(time.perf_counter() - t0)
        for name, events in new_events.items():
            for event in events:
                if event in ("ENTRY", "ENTRY_RECOVERED", "INITIAL", "EXIT", "LOST"):
                    key = name.replace(" ", "_") + "_" + event
                    if key not in snapshots:
                        snapshots[key] = big.copy()
        for name, c in counters.items():
            estimate, start_estimate = c.estimate(), c.wait_to_start()
            timeline.append({"frame": f, "time_s": f / FPS, "lane": name,
                "queue": c.queue_len(), "waiting": c.waiting_len(), "serving": int(c.serving_id is not None),
                "entries": c.entries, "initial": c.initial, "recovered": c.recovered,
                "exits": c.exits, "lost": c.lost, "n_samples": len(c.service_samples),
                "estimate_source": c.estimate_source(),
                "wait_start_mid_s": start_estimate[1] if start_estimate is not None else np.nan,
                "est_lo_s": estimate[0] if estimate is not None else np.nan,
                "est_mid_s": estimate[1] if estimate is not None else np.nan,
                "est_hi_s": estimate[2] if estimate is not None else np.nan,
                "eta_video_s": f / FPS + estimate[1] if estimate is not None else np.nan,
                "n_tracks": len(tracks)})
        if SAVE_TRACK_LOG:
            for raw, box in tracks:
                state, counter = person_state(counters, raw)
                px, py = foot_point(box)
                track_log.append({"frame": f, "time_s": f / FPS, "raw_id": raw,
                    "canonical_id": counter.alias.get(raw, raw) if counter is not None else raw,
                    "lane": counter.name if counter is not None else "", "state": state,
                    "foot_x": px, "foot_y": py, "x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3]})
        f += 1
        if f % max(1, int(15 * FPS)) == 0:
            print(f"Proses {f}/{N_FRAMES or '?'} frame | waktu video {fmt_clock(f / FPS)} | FPS proses {1 / np.mean(latencies[-30:]):.1f}")
finally:
    cap.release()
    writer.release()

if not timeline:
    raise RuntimeError("Tidak ada frame diproses. Periksa VIDEO_PATH dan MAX_FRAMES.")
for name, c in counters.items():
    print(f"{name}: masuk={c.entries} (pulih={c.recovered}), awal={c.initial}, keluar={c.exits}, hilang={c.lost}, sisa={c.queue_len()}, ID digabung={c.merged}")
    print("  Jika bergabung sekarang:", fmt_range(c.estimate()), "|", c.estimate_source())
    if c.entries + c.initial == 0:
        print("  Tidak ada pelanggan terdaftar: periksa titik kaki/zona pada preview dan track_log.csv.")
print(f"Selesai: {f} frame. Akhir video tidak otomatis dianggap EXIT/LOST.")


## 9. Hasil, data pelanggan, dan grafik waktu
Hasil yang diekspor: `event_log.csv`, `timeline.csv`, `pelanggan.csv`, `ringkasan_kasir.csv`, ringkasan per jendela waktu, tes logika, dan `track_log.csv` untuk memeriksa posisi kaki/ID.

`pelanggan.csv` memisahkan waktu masuk yang diketahui dan waktu mulai terlihat. Orang dengan `INITIAL`/`RECOVERED` tidak diberi waktu masuk buatan. Tabel ringkasan kasir menjawab: **jika seseorang masuk antrean sekarang, berapa lama sampai mulai dilayani dan sampai selesai?**


In [ ]:
EVENT_COLUMNS = ["lane", "frame", "time_s", "event", "track_id", "queue", "source", "reason", "entry_time_known",
                 "wait_s", "service_s", "service_sample_s", "sample_source", "est_lo", "est_mid", "est_hi", "merged_into", "partial"]
EVENTS = pd.DataFrame([event for c in counters.values() for event in c.events]).reindex(columns=EVENT_COLUMNS)
if not EVENTS.empty:
    EVENTS = EVENTS.sort_values(["frame", "lane"], kind="stable").reset_index(drop=True)
TIMELINE = pd.DataFrame(timeline)
TRACKS = pd.DataFrame(track_log, columns=["frame", "time_s", "raw_id", "canonical_id", "lane", "state", "foot_x", "foot_y", "x1", "y1", "x2", "y2"])
PEOPLE_COLUMNS = ["lane", "track_id", "status", "entry_source", "entry_time_known", "observed_from_s", "entry_video_s",
                  "service_start_video_s", "exit_video_s", "total_observed_s", "service_observed_s", "service_partial", "est_at_entry_mid_s"]
people = []
for name, c in counters.items():
    for tid, record in c.records.items():
        start = record["service_start_frame"]
        estimate = record["est_at_entry"]
        people.append({"lane": name, "track_id": tid, "status": c.state_of(tid),
            "entry_source": record["entry_source"], "entry_time_known": record["entry_known"],
            "observed_from_s": record["observed_frame"] / FPS,
            "entry_video_s": record["entry_frame"] / FPS if record["entry_known"] else np.nan,
            "service_start_video_s": start / FPS if start is not None else np.nan,
            "exit_video_s": record.get("exit_frame", np.nan) / FPS,
            "total_observed_s": record.get("total_observed_s", c.elapsed(tid)),
            "service_observed_s": record.get("service_s", np.nan),
            "service_partial": record["service_partial"],
            "est_at_entry_mid_s": estimate[1] if estimate is not None else np.nan})
PEOPLE = pd.DataFrame(people, columns=PEOPLE_COLUMNS)

summary = []
for name, c in counters.items():
    estimate, wait = c.estimate(), c.wait_to_start()
    summary.append({"lane": name, "masuk_garis": c.entries - c.recovered, "masuk_pemulihan": c.recovered,
        "sudah_antre_di_awal": c.initial, "keluar": c.exits, "hilang": c.lost,
        "menunggu": c.waiting_len(), "dilayani": int(c.serving_id is not None), "total_dalam_antrean": c.queue_len(),
        "mulai_dilayani_s": wait[1] if wait is not None else np.nan,
        "selesai_median_s": estimate[1] if estimate is not None else np.nan,
        "rentang_selesai": fmt_range(estimate), "sumber_estimasi": c.estimate_source(),
        "sampel_layanan": len(c.service_samples), "id_digabung": c.merged})
SUMMARY = pd.DataFrame(summary)

TL = TIMELINE.sort_values(["lane", "frame"]).copy()
TL["window"] = (TL["time_s"] // WINDOW_S).astype(int)
for src, dst in (("entries", "d_in"), ("exits", "d_out"), ("recovered", "d_recovered"), ("initial", "d_initial")):
    TL[dst] = TL.groupby("lane")[src].diff().fillna(TL[src])
WIN = TL.groupby(["lane", "window"]).agg(
    antre_rata2=("queue", "mean"), antre_maks=("queue", "max"), menunggu_rata2=("waiting", "mean"),
    masuk=("d_in", "sum"), keluar=("d_out", "sum"), masuk_pemulihan=("d_recovered", "sum"), awal=("d_initial", "sum"),
    mulai_dilayani_s=("wait_start_mid_s", "mean"), selesai_bawah_s=("est_lo_s", "mean"),
    selesai_median_s=("est_mid_s", "mean"), selesai_atas_s=("est_hi_s", "mean"),
).reset_index()
WIN["rentang_video"] = [f"{fmt_clock(w * WINDOW_S)}-{fmt_clock((w + 1) * WINDOW_S)}" for w in WIN["window"]]
WIN["rentang_estimasi_selesai"] = [fmt_range((lo, mid, hi)) if pd.notna(mid) else "kalibrasi"
    for lo, mid, hi in zip(WIN.selesai_bawah_s, WIN.selesai_median_s, WIN.selesai_atas_s)]

for filename, table in (("event_log.csv", EVENTS), ("timeline.csv", TIMELINE), ("pelanggan.csv", PEOPLE),
                        ("ringkasan_kasir.csv", SUMMARY), ("ringkasan_per_jendela_waktu.csv", WIN), ("unit_tests.csv", TEST_DF)):
    table.to_csv(OUT / filename, index=False, encoding="utf-8-sig")
if SAVE_TRACK_LOG:
    TRACKS.to_csv(OUT / "track_log.csv", index=False, encoding="utf-8-sig")

print("ESTIMASI BAGI PELANGGAN YANG BARU AKAN BERGABUNG (kondisi akhir video):")
display(SUMMARY)
print("Ringkasan per rentang waktu video:")
display(WIN.drop(columns="window").round(2))
if EVENTS.empty:
    print("Belum ada event. Periksa preview titik kaki dan posisi zona pelanggan.")
else:
    print(EVENTS.groupby(["lane", "event"]).size().to_string())

for name, img in snapshots.items():
    cv2.imwrite(str(OUT / f"screenshots/03_event_{name}.jpg"), img)
    show(img, name)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for name, group in TIMELINE.groupby("lane"):
    t = group["time_s"].to_numpy(float)
    axes[0].step(t, group["queue"], where="post", label=name)
    axes[1].plot(t, group["est_mid_s"].to_numpy(float), label=name)
    axes[1].fill_between(t, group["est_lo_s"].to_numpy(float), group["est_hi_s"].to_numpy(float), alpha=0.18)
    axes[2].plot(t, group["wait_start_mid_s"].to_numpy(float), label=name)
axes[0].set(title="Orang dalam sistem (menunggu + sedang dilayani)", ylabel="orang")
axes[1].set(title="Jika bergabung sekarang: estimasi sampai selesai", ylabel="detik")
axes[2].set(title="Jika bergabung sekarang: estimasi sampai mulai dilayani", ylabel="detik", xlabel="waktu video (detik)")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUT / "screenshots/04_grafik_antrean_dan_waktu.png", dpi=130)
plt.show()

run_metadata = {"video": str(SRC), "width": W, "height": H, "fps_video": FPS, "processed_frames": f,
    "model": MODEL_NAME, "conf": CONF, "imgsz": IMGSZ, "tracker": Path(TRACKER_USED).name,
    "prior_range_s": SERVICE_PRIOR_RANGE_S, "bootstrap_s": BOOTSTRAP_S, "roi_recovery": RECOVERY_ENABLED,
    "exit_deadband_px": EXIT_DEADBAND_PX, "recovery_exit_guard_px": RECOVERY_EXIT_GUARD_PX,
    "outside_roi_timeout_s": OUTSIDE_ROI_TIMEOUT_S,
    "lanes_reference": LANES, "lanes_video": ACTIVE_LANES,
    "notes": "Rentang deskriptif/perkiraan, bukan interval kepercayaan. INITIAL/RECOVERED bukan pengamatan crossing entry."}
(OUT / "run_config.json").write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False))


## 10. Evaluasi
MAE estimasi sampai selesai hanya dihitung pada pasangan entry–exit yang lengkap, dengan waktu masuk benar-benar teramati. Pengamatan awal yang terpotong tidak diperlakukan sebagai durasi penuh.

Isi `GT` dari hitungan manual pada rekaman dan rentang frame yang sama sebelum melaporkan kesalahan counting video. Selisih jumlah bukan metrik deteksi/identitas seperti precision, recall, atau IDF1.


In [ ]:
lat = np.asarray(latencies, dtype=float)
if lat.size:
    processing_fps = 1.0 / lat.mean()
    print(f"FPS end-to-end: {processing_fps:.2f}; latensi median {np.median(lat)*1000:.0f} ms; p95 {np.percentile(lat,95)*1000:.0f} ms")
    print(f"FPS input: {FPS:.2f}; {'mencapai' if processing_fps >= FPS else 'belum mencapai'} kecepatan video pada perangkat ini.")
print("Durasi antre dan ETA memakai waktu VIDEO, bukan waktu inferensi komputer.")

# Validasi total durasi hanya untuk orang yang crossing masuknya benar-benar diamati.
exits = EVENTS.loc[EVENTS["event"] == "EXIT"].copy()
valid = exits.loc[exits["entry_time_known"].eq(True)].dropna(subset=["wait_s", "est_lo", "est_mid", "est_hi"]).copy()
if not valid.empty:
    valid["abs_error_s"] = (valid["wait_s"] - valid["est_mid"]).abs()
    valid["di_dalam_rentang"] = valid["wait_s"].between(valid["est_lo"], valid["est_hi"])
    print(f"MAE estimasi sampai selesai: {valid.abs_error_s.mean():.2f} detik, {len(valid)} pelanggan.")
    print(f"Aktual berada dalam rentang: {valid.di_dalam_rentang.mean():.1%} (bukan akurasi deteksi orang).")
    display(valid[["lane", "track_id", "wait_s", "est_lo", "est_mid", "est_hi", "abs_error_s", "di_dalam_rentang"]].round(2))
else:
    print("Belum cukup pasangan entry lengkap + exit + estimasi untuk menghitung MAE.")
print("INITIAL/RECOVERED: durasi sebelum pertama terlihat tidak diketahui; total durasi yang dicatat adalah batas bawah pengamatan.")

# Isi setelah menghitung crossing garis secara manual pada VIDEO dan rentang frame yang sama.
GT = None
# Contoh format (angka hanya contoh, jangan dipakai sebagai hasil):
# GT = {"Kasir Tengah": {"masuk_garis": 3, "keluar": 4}, "Kasir Kanan": {"masuk_garis": 2, "keluar": 3}}
if GT is None:
    print("Ground truth manual belum diisi; akurasi counting video belum dinyatakan.")
else:
    for name, c in counters.items():
        for label, predicted in (("masuk_garis", c.entries - c.recovered), ("keluar", c.exits)):
            truth = GT[name][label]
            print(f"{name} {label}: prediksi={predicted}, manual={truth}, selisih absolut={abs(predicted-truth)}")
print("\nTes logika:", int(TEST_DF.hasil.eq("PASS").sum()), "/", len(TEST_DF), "PASS")
print("Exit tanpa registrasi:", int(EVENTS.event.eq("EXIT_IGNORED_NOT_REGISTERED").sum()))


## 11. Putar demo dan ambil paket hasil
OpenCV menulis MP4 mentah. Jika `ffmpeg` tersedia, cell ini membuat H.264 agar dapat diputar di browser. ZIP berisi video, CSV, konfigurasi run, dan screenshot. Cell ini tidak mengunggah atau mempublikasikan hasil ke layanan lain.


In [ ]:
import shutil
import subprocess
import zipfile
from IPython.display import Video, FileLink

DEMO = OUT / "queuewise_demo_h264.mp4"
if not RAW.exists() or RAW.stat().st_size == 0:
    raise RuntimeError("Video output belum tersedia; jalankan pipeline utama dahulu")

def to_h264(src, dst):
    if shutil.which("ffmpeg") is None:
        return False
    result = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
        "-c:v", "libx264", "-crf", "24", "-preset", "veryfast", "-pix_fmt", "yuv420p",
        "-movflags", "+faststart", str(dst)], capture_output=True, text=True)
    if result.returncode != 0:
        print("Konversi H.264 gagal:", result.stderr[-500:])
    return result.returncode == 0 and dst.exists() and dst.stat().st_size > 0

if to_h264(RAW, DEMO):
    display(Video(str(DEMO), embed=True, width=1100))
    print("Video demo:", DEMO)
else:
    print("H.264 belum tersedia. File MP4 mentah tetap tersimpan dan bisa diputar lewat pemutar video lokal.")
    cap = cv2.VideoCapture(str(RAW))
    try:
        indices = np.linspace(0, max(0, f - 1), 4).astype(int)
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, fr = cap.read()
            if ok:
                show(fr, f"Cuplikan t={idx / FPS:.1f} s")
    finally:
        cap.release()

ZIP_PATH = OUT.parent / "QueueWise_hasil_demo.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUT.parent))
display(FileLink(str(ZIP_PATH)))
print("Paket hasil berisi video, event, timeline, data pelanggan, estimasi waktu, track debug, dan screenshot.")


## 12. Batas interpretasi dan langkah pemeriksaan
- Zona dipilih dari frame yang tersedia dalam notebook untuk sudut kamera ini; perlu dicek pada gerak pelanggan sepanjang video asli. Perubahan kamera memerlukan kalibrasi ulang.
- Orang yang lewat/berhenti di koridor pelanggan dapat dianggap pelanggan oleh registrasi zona. Area petugas sengaja dipisahkan, tetapi deteksi bbox yang terpotong tetap bisa memindahkan titik kaki.
- Pemulihan ID memakai jarak, ukuran bbox, dan waktu. Dua kandidat yang terlalu mirip tidak digabung otomatis, tetapi identitas masih dapat tertukar saat occlusion berat.
- Label `SERVING` memakai lokasi dan urutan FIFO, bukan pengenalan transaksi pembayaran. Sampel waktunya adalah pendekatan visual sampai garis keluar.
- `INITIAL` dan `ENTRY_RECOVERED` adalah pemulihan keadaan antrean. Tampilkan jumlahnya terpisah jika membandingkan crossing masuk terhadap hitungan manual.
- Kalibrasi waktu dapat tetap belum cukup pada video sangat sepi atau bila awal layanan selalu tertutup. Dalam kondisi itu sumber tetap **asumsi awal/manual**, atau **kalibrasi** bila prior dimatikan.
- Jumlah barang, pergantian petugas, orang meninggalkan antrean melalui sisi lain, dan penutupan kasir belum dimodelkan. Estimasi dapat berubah; ia bukan janji waktu penyelesaian.

**Pemeriksaan demo:** tonton video output sambil membandingkan event dan titik kaki pada `track_log.csv`; hitung crossing secara manual; baru simpulkan akurasi pada rekaman tersebut.

**Referensi implementasi tracker:** [Ultralytics — Multi-Object Tracking](https://docs.ultralytics.com/modes/track/), [default BoT-SORT](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/trackers/botsort.yaml).
